# Table 10.1 Data-Point Editor
Edit the running-example points and update Figure 10.2 plus Tables 10.2 to 10.5 automatically.


In [ ]:
import math
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

DEFAULT_POINTS = [
    [0.0, 2.0],
    [2.0, 0.0],
    [3.0, 1.0],
    [5.0, 1.0],
]


def fmt(value):
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return f"{value:.3f}".rstrip('0').rstrip('.')


def pairwise(points, metric):
    n = len(points)
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            dx = abs(points[i][0] - points[j][0])
            dy = abs(points[i][1] - points[j][1])
            if metric == 'euclidean':
                value = math.hypot(dx, dy)
            elif metric == 'manhattan':
                value = dx + dy
            elif metric == 'linf':
                value = max(dx, dy)
            elif metric == 'l0':
                value = float((dx != 0) + (dy != 0))
            else:
                raise ValueError(metric)
            mat[i, j] = value
            mat[j, i] = value
    return mat


def closest_pair(matrix):
    best_value = None
    best_pair = (0, 1)
    n = matrix.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            value = matrix[i, j]
            if best_value is None or value < best_value:
                best_value = value
                best_pair = (i, j)
    return best_pair, best_value


def sample_table_html(points):
    rows = []
    for i, (xv, yv) in enumerate(points, start=1):
        rows.append(
            f"<tr><td><b>x{i}</b></td><td>{fmt(xv)}</td><td>{fmt(yv)}</td></tr>"
        )
    body = ''.join(rows)
    return (
        '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
        '<thead><tr>'
        '<th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">sample</th>'
        '<th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">x_{i,1}</th>'
        '<th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">x_{i,2}</th>'
        '</tr></thead>'
        f'<tbody>{body}</tbody>'
        '</table>'
    )


def distance_table_html(matrix, title_label):
    n = matrix.shape[0]
    header = ''.join(
        f'<th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">x{i}</th>'
        for i in range(1, n + 1)
    )
    rows = []
    for i in range(n):
        cells = ''.join(
            f'<td style="border:1px solid #cbd5e1;padding:7px 8px;text-align:center;">{fmt(matrix[i, j])}</td>'
            for j in range(n)
        )
        rows.append(
            f'<tr><th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">x{i + 1}</th>{cells}</tr>'
        )
    body = ''.join(rows)
    return (
        '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
        '<thead><tr>'
        f'<th style="border:1px solid #cbd5e1;padding:7px 8px;background:#dbeafe;">{title_label}</th>'
        f'{header}'
        '</tr></thead>'
        f'<tbody>{body}</tbody>'
        '</table>'
    )


def point_box(name, x_value, y_value):
    title = widgets.HTML(value=f'<b>{name}</b>')
    x_widget = widgets.FloatText(value=x_value, description='x', step=0.1, layout=widgets.Layout(width='140px'))
    y_widget = widgets.FloatText(value=y_value, description='y', step=0.1, layout=widgets.Layout(width='140px'))
    box = widgets.VBox([title, x_widget, y_widget], layout=widgets.Layout(border='1px solid #cbd5e1', padding='10px', border_radius='12px', width='170px'))
    return x_widget, y_widget, box


point_widgets = []
point_boxes = []
for idx, (xv, yv) in enumerate(DEFAULT_POINTS, start=1):
    xw, yw, box = point_box(f'x{idx}', xv, yv)
    point_widgets.append((xw, yw))
    point_boxes.append(box)

reset_btn = widgets.Button(description='Reset default points', button_style='')
plot_out = widgets.Output()
sample_html = widgets.HTML()
euclidean_html = widgets.HTML()
manhattan_html = widgets.HTML()
linf_html = widgets.HTML()
l0_html = widgets.HTML()
status_html = widgets.HTML()


def current_points():
    return [[float(xw.value), float(yw.value)] for xw, yw in point_widgets]


def table_card(title, html_widget, caption):
    header = widgets.HTML(value=f'<b>{title}</b>')
    cap = widgets.HTML(value=f'<div style="margin-top:8px;color:#475569;font-size:13px;">{caption}</div>')
    return widgets.VBox(
        [header, html_widget, cap],
        layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', border_radius='14px')
    )


def render(*_):
    points = current_points()
    euclidean = pairwise(points, 'euclidean')
    manhattan = pairwise(points, 'manhattan')
    linf = pairwise(points, 'linf')
    l0 = pairwise(points, 'l0')

    sample_html.value = sample_table_html(points)
    euclidean_html.value = distance_table_html(euclidean, 'r = 2')
    manhattan_html.value = distance_table_html(manhattan, 'r = 1')
    linf_html.value = distance_table_html(linf, 'r = infinity')
    l0_html.value = distance_table_html(l0, 'r = 0')

    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    labels = [f'x{i}' for i in range(1, len(points) + 1)]
    margin = 1.0
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=xs,
        y=ys,
        mode='markers+text',
        text=labels,
        textposition='top center',
        marker=dict(size=12, color='#2563eb'),
        hovertemplate='%{text} = (%{x}, %{y})<extra></extra>'
    ))
    fig.update_layout(
        title='Figure 10.2: Running example points',
        height=500,
        margin=dict(l=60, r=20, t=50, b=55),
        paper_bgcolor='#ffffff',
        plot_bgcolor='#ffffff',
        showlegend=False,
    )
    fig.update_xaxes(title='x_1', range=[min(xs) - margin, max(xs) + margin], gridcolor='#e2e8f0', zeroline=True)
    fig.update_yaxes(title='x_2', range=[min(ys) - margin, max(ys) + margin], gridcolor='#e2e8f0', zeroline=True, scaleanchor='x', scaleratio=1)

    with plot_out:
        plot_out.clear_output(wait=True)
        fig.show()

    pair_e, value_e = closest_pair(euclidean)
    pair_m, value_m = closest_pair(manhattan)
    pair_i, value_i = closest_pair(linf)
    pair_0, value_0 = closest_pair(l0)
    status_html.value = (
        '<div style="white-space:pre-wrap;line-height:1.55;">'
        f'Closest pair under Euclidean: x{pair_e[0] + 1} and x{pair_e[1] + 1} (d = {fmt(value_e)})\n'
        f'Closest pair under Manhattan: x{pair_m[0] + 1} and x{pair_m[1] + 1} (d = {fmt(value_m)})\n'
        f'Closest pair under L_infinity: x{pair_i[0] + 1} and x{pair_i[1] + 1} (d = {fmt(value_i)})\n'
        f'Closest pair under L_0: x{pair_0[0] + 1} and x{pair_0[1] + 1} (d = {fmt(value_0)})'
        '</div>'
    )


for xw, yw in point_widgets:
    xw.observe(render, names='value')
    yw.observe(render, names='value')


def on_reset(_):
    for (xw, yw), (xv, yv) in zip(point_widgets, DEFAULT_POINTS):
        xw.value = xv
        yw.value = yv


reset_btn.on_click(on_reset)

controls = widgets.VBox([
    widgets.HTML(value='<h2 style="margin:0 0 8px;">Interactive Table 10.1 Data-Point Editor</h2><div style="color:#475569;margin-bottom:10px;">Edit the four running-example samples and the derived running-example figure plus Tables 10.2 to 10.5 update automatically.</div>'),
    widgets.HBox(point_boxes, layout=widgets.Layout(flex_flow='row wrap', gap='12px')),
    reset_btn,
])

plot_card = widgets.VBox([
    plot_out,
    widgets.HTML(value='<div style="margin-top:8px;color:#475569;font-size:13px;">Figure 10.2. Visualization of the running-example points used throughout Section 10.2.</div>')
], layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', border_radius='14px'))

row1 = widgets.HBox([
    table_card('Table 10.1', sample_html, 'Running example used to compare proximity measures.'),
    table_card('Table 10.2', euclidean_html, 'Pairwise Euclidean distances for the running example.'),
], layout=widgets.Layout(gap='16px'))
row2 = widgets.HBox([
    table_card('Table 10.3', manhattan_html, 'Pairwise Manhattan distances for the running example.'),
    table_card('Table 10.4', linf_html, 'L_infinity distances on the running example.'),
], layout=widgets.Layout(gap='16px'))
row3 = widgets.HBox([
    table_card('Table 10.5', l0_html, 'L_0 coordinate difference counts on the running example.'),
], layout=widgets.Layout(gap='16px'))
status_card = widgets.VBox([
    widgets.HTML(value='<b>Summary</b>'),
    status_html,
], layout=widgets.Layout(border='1px solid #cbd5e1', padding='12px', border_radius='14px'))

render()
display(widgets.VBox([controls, plot_card, row1, row2, row3, status_card], layout=widgets.Layout(gap='16px')))
